# 03 — Products: near-optimality (D11/D30), the counterfactual width (D17), route branches (D12/D26) and the link classes (D23–D25) → `finish`

Re-attaches to the run (`cc.load` + `cost_distances` cache HIT + `corridor_network` from the band store), then the
products the classes and maps need, IN THIS ORDER (the northern 04 after the 2026-09-28 rules):

1. **Near-optimality surface** (D11; fixed slack breaks at cutoff/6, cutoff/2, cutoff — D30; **G10**, **G22**).
2. **The counterfactual width** (D17): a SECOND cost-weighted-distance set on a surface with every cost ≥ 10 relaxed to 1
   (compact cache, resumable, ~the notebook-02 CWD runtime), every link banded on it, and per link the barrier-free
   median cross-section, the least-cost path length in cells, the tenth-percentile width ratio (D28) and the barrier flag.
   It now decides two geometric rules BEFORE any corridor class: links too short for the width test (D24) and
   near-contiguous links (D25, path shorter than the barrier-free width — a blob, not a route). **G13**.
3. **Route branches** (D12) with the relative sliver floor (D26; **G9**, **G21**) — near-contiguous links get none — and then
   `classify_links`: the ONE derivation of every link's class (D23 precedence: the top class needs no alternative link, one
   branch AND a corridor below its barrier-free width; **G18** eight-cell table → `class_truth_table.csv`; **G19** floor effect →
   `floor_effect.csv`).
4. Protection status (W11), `finish`.

Disk: the counterfactual cache is another ~8 GB compact set under `input_data/corridors_300m_y2y/cwd_cache/<sha>_cf_c/`.

In [ ]:
# ---- Setup: find the project root, import the shared engines ----------------
# This notebook lives in analyses/wolverine_refugia_connectivity/, below the repo root where
# config.py and the corridor engine modules sit. Same bootstrap as analyses/northern_connectivity/.
import sys, pathlib, json
_cands = [p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
          if (p / "config.py").exists()]
assert _cands, f"config.py not found above {pathlib.Path.cwd()} -- run this notebook from inside the repo"
ROOT = _cands[0]
sys.path.insert(0, str(ROOT))

import importlib
import numpy as np
import pandas as pd
import config
import corridors_prep as cp
import corridor_graph as cg
import corridors_core as cc
for _m in (config, cp, cg, cc):
    importlib.reload(_m)

KEY = "wolverine"
CFG = config.CORRIDORS[KEY]
HERE = ROOT / "analyses" / "wolverine_refugia_connectivity"

In [ ]:
RUN = "v2_run001"          # <- the run created by notebook 02 (see cc.runs(KEY))
A = cc.load(config.RESULTS_DIR / CFG["results_subdir"] / RUN)
cc.resistance(A)
cc.cost_distances(A)                        # cache HIT
cc.corridor_network(A, verbose=False)       # bands from the store: seconds
if A.cfg.get("cutoff_detour_km") is None:   # D31 twin for a run created before the constant existed (record only; G22 checks the pair)
    A.cfg["cutoff_detour_km"] = A.rec["cfg"]["cutoff_detour_km"] = float(A.cutoff) * A.cell_km
    (A.run_dir / "run_config.json").write_text(json.dumps(A.rec, indent=2, ensure_ascii=False))
    print(f"  D31: cutoff_detour_km {A.cfg['cutoff_detour_km']:.4f} km pinned into run_config.json")
print(f"{A.run_id}: {len(A.edges)} edges, corridor {int(A.corridor.sum())*A.cell_km2:,.0f} km² | band = "
      f"{A.cfg['cwd_cutoff_abs']:.4g} cost units = about {A.cfg['cwd_cutoff_abs']*A.cell_km:.1f} km of extra travel on open ground (D31)")

## 1 · Near-optimality surface (**G10**, **G22**) — fixed cost-unit breaks (D30)

In [ ]:
cc.near_optimality(A)

## 2 · The counterfactual width (**D17**, **G13**) — the second CWD set

Runs BEFORE the branches: it measures the barrier-free width that decides the near-contiguous class (D25) and the
resolution floor (D24). Free disk is checked first (≥ 10 GB).

In [ ]:
import shutil
free_gb = shutil.disk_usage(ROOT).free / 1e9
assert free_gb >= 10, f"only {free_gb:.1f} GB free -- the counterfactual cache needs ~8 GB; free some disk first"
cc.counterfactual_squeeze(A)

## 3 · Route branches (**D12**, relative floor **D26**; **G9**, **G21**) → the link classes (**D23–D25**; **G18**, **G19**)

In [ ]:
cc.route_branches(A)

The eight-cell table (edge-irreplaceable × one-branch × narrow) and the floor-effect diagnostic, as written to the run dir.
On a dense refugia network the geometric classes decided before any corridor class are most of the answer to "why so few
corridors on the map": their counts lead the report.

In [ ]:
tt = pd.read_csv(A.run_dir / "class_truth_table.csv"); fe = pd.read_csv(A.run_dir / "floor_effect.csv")
print("eight-cell table (E = no alternative link, B1 = one branch, S = narrow):"); print(tt.to_string(index=False))
print("\nfloor effect (registered vs halved / doubled floors):"); print(fe.to_string(index=False))
A.edges["link_class_label"].value_counts()

## 4 · Protection status (W11) re-derived on the classified network, then finish

In [ ]:
cc.secured_status(A)
cc.priority_surface(A)
cc.finish(A)
cc.runs(KEY)

## Done — next: `04_tables_and_figures.ipynb` (the record), `05_v2_postprocess.ipynb` (the v2.5 product), then `06_director_outputs.ipynb` (the curated set).